# 💸 Japan Grid Analysis: Curtailment & Monetization

**Notebook 04:** Quantify curtailed renewable energy and calculate its economic value.

This notebook answers: **How much** curtailed energy could be monetized?

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# Load cleaned data
df = pd.read_csv('../data/processed/japan_grid_clean.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

print(f"✅ Data loaded: {len(df):,} records")

## Calculate Curtailment Metrics

In [ ]:
# Curtailment analysis by region
curtailment_analysis = []

for region in sorted(df['region'].unique()):
    region_df = df[df['region'] == region]
    
    # Only calculate for surplus hours
    surplus_hours = region_df[region_df['surplus_mw'] > 0]
    
    if len(surplus_hours) > 0:
        curtailed_mwh = surplus_hours['surplus_mw'].sum()  # 1-hour intervals = MW to MWh
        avg_surplus = surplus_hours['surplus_mw'].mean()
        avg_price = surplus_hours['price_jpy_kwh'].mean()
        curtailed_value_jpy = curtailed_mwh * 1000 * avg_price  # Convert MWh to kWh
        USD_JPY_RATE = 0.0067
        curtailed_value_usd = curtailed_value_jpy * USD_JPY_RATE
        surplus_hours_count = len(surplus_hours)
        
        curtailment_analysis.append({
            'Region': region,
            'Surplus Hours': surplus_hours_count,
            'Curtailed MWh': curtailed_mwh,
            'Avg Surplus MW': avg_surplus,
            'Avg Price JPY': avg_price,
            'Value JPY': curtailed_value_jpy,
            'Value USD': curtailed_value_usd
        })

curtailment_df = pd.DataFrame(curtailment_analysis).sort_values('Curtailed MWh', ascending=False)

print("\n📊 CURTAILMENT ANALYSIS BY REGION:")
print(curtailment_df.to_string(index=False))

print(f"\n\n🌍 JAPAN-WIDE TOTALS:")
print(f"Total Curtailed MWh: {curtailment_df['Curtailed MWh'].sum():,.0f}")
print(f"Total Value JPY: ¥{curtailment_df['Value JPY'].sum():,.0f}")
print(f"Total Value USD: ${curtailment_df['Value USD'].sum():,.0f}")

## Chart 1: Curtailment Volume by Region

In [ ]:
# Sort for horizontal bar chart
curtailment_sorted = curtailment_df.sort_values('Curtailed MWh', ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=curtailment_sorted['Region'],
    x=curtailment_sorted['Curtailed MWh'],
    orientation='h',
    marker=dict(
        color=curtailment_sorted['Curtailed MWh'],
        colorscale='Greens',
        showscale=False
    ),
    text=curtailment_sorted['Curtailed MWh'].round(0),
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Curtailed: %{x:,.0f} MWh<extra></extra>'
))

fig.update_layout(
    title="Annual Renewable Curtailment by Region (MWh)",
    xaxis_title="Curtailed Energy (MWh)",
    yaxis_title="Region",
    height=500,
    template='plotly_white',
    showlegend=False
)

fig.show()
print("✅ Curtailment volumes chart generated")

## Chart 2: Price vs Surplus Correlation

In [ ]:
# Create scatter: surplus vs price (all data points, colored by region)
# Sample data if too large for performance
plot_df = df.sample(n=min(5000, len(df)), random_state=42)

fig = go.Figure()

# Add scatter for each region
for region in sorted(plot_df['region'].unique()):
    region_data = plot_df[plot_df['region'] == region]
    fig.add_trace(go.Scatter(
        x=region_data['surplus_mw'],
        y=region_data['price_jpy_kwh'],
        mode='markers',
        name=region,
        marker=dict(size=4, opacity=0.6)
    ))

# Add trend line
z = np.polyfit(plot_df['surplus_mw'], plot_df['price_jpy_kwh'], 1)
p = np.poly1d(z)
x_trend = np.linspace(plot_df['surplus_mw'].min(), plot_df['surplus_mw'].max(), 100)
fig.add_trace(go.Scatter(
    x=x_trend,
    y=p(x_trend),
    mode='lines',
    name='Trend',
    line=dict(color='red', width=3, dash='dash')
))

fig.update_layout(
    title="JEPX Spot Price vs Renewable Surplus<br><sub>Lower prices = higher surplus = ideal for flexible computing demand</sub>",
    xaxis_title="Renewable Surplus (MW)",
    yaxis_title="JEPX Spot Price (¥/kWh)",
    height=600,
    template='plotly_white',
    hovermode='closest'
)

fig.show()

# Calculate correlation
correlation = df['surplus_mw'].corr(df['price_jpy_kwh'])
print(f"\n✅ Correlation between surplus and price: {correlation:.3f}")
print(f"(Negative correlation = inverse relationship: ↑ surplus → ↓ price)")

## Chart 3: Distributed Computing Revenue Opportunity

In [ ]:
# Mining container characteristics
MINER_POWER_KW = 3.25  # Single miner power draw
MINING_REVENUE_PER_HOUR_USD = 2.50  # Conservative estimate

# Calculate opportunity for each region
opportunity_data = []

for region in sorted(df['region'].unique()):
    region_df = df[df['region'] == region]
    surplus_df = region_df[region_df['surplus_mw'] > 0]
    
    if len(surplus_df) > 0:
        # How many miners could run on this surplus?
        avg_surplus_mw = surplus_df['surplus_mw'].mean()
        miners_supportable = avg_surplus_mw * 1000 / MINER_POWER_KW
        
        # Potential annual revenue
        surplus_hours = len(surplus_df)
        potential_annual_revenue_usd = miners_supportable * surplus_hours * MINING_REVENUE_PER_HOUR_USD
        
        opportunity_data.append({
            'Region': region,
            'Avg Surplus MW': avg_surplus_mw,
            'Surplus Hours': surplus_hours,
            'Miners Supportable': miners_supportable,
            'Annual Revenue USD': potential_annual_revenue_usd
        })

opportunity_df = pd.DataFrame(opportunity_data).sort_values('Annual Revenue USD', ascending=False)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=opportunity_df['Region'],
    y=opportunity_df['Annual Revenue USD'],
    marker=dict(
        color=opportunity_df['Annual Revenue USD'],
        colorscale='Viridis',
        showscale=False
    ),
    text=opportunity_df['Annual Revenue USD'].apply(lambda x: f"${x:,.0f}"),
    textposition='outside',
    hovertemplate='<b>%{x}</b><br>Potential Revenue: $%{y:,.0f}<extra></extra>'
))

fig.update_layout(
    title="Distributed Computing Revenue Potential by Region<br><sub>Based on mining when renewable surplus occurs</sub>",
    xaxis_title="Region",
    yaxis_title="Potential Annual Revenue (USD)",
    height=500,
    template='plotly_white',
    showlegend=False
)
fig.update_xaxes(tickangle=-45)

fig.show()

print(f"\n📊 OPPORTUNITY SIZING:")
print(opportunity_df.to_string(index=False))
print(f"\nTotal Japan-wide opportunity: ${opportunity_df['Annual Revenue USD'].sum():,.0f}/year")

## 🔍 Key Curtailment Insights

In [ ]:
top_3_regions = curtailment_df.head(3)

print("\n🎯 TOP 3 CURTAILMENT OPPORTUNITY REGIONS:")
for i, (_, row) in enumerate(top_3_regions.iterrows(), 1):
    print(f"\n{i}. {row['Region']}")
    print(f"   • Curtailable energy: {row['Curtailed MWh']:,.0f} MWh/year")
    print(f"   • Market value: ${row['Value USD']:,.0f}/year")
    print(f"   • Surplus hours: {row['Surplus Hours']:,}")
    print(f"   • Average spot price: ¥{row['Avg Price JPY']:.2f}/kWh")

print("\n" + "="*70)
print("\n💡 KEY FINDINGS:")
print(f"\n1. MARKET SIZE: ¥{curtailment_df['Value JPY'].sum():,.0f}/year (~${curtailment_df['Value USD'].sum():,.0f})")
print(f"\n2. PRICE SIGNAL: Negative correlation (r={correlation:.3f}) confirms:")
print(f"   When surplus ↑ → Spot prices ↓ → Mining profitability ↑")
print(f"\n3. DEPLOYMENT OPPORTUNITY: Top 3 regions represent {(top_3_regions['Curtailed MWh'].sum() / curtailment_df['Curtailed MWh'].sum() * 100):.1f}% of total curtailment")
print(f"\n4. MINING CAPACITY: ~{opportunity_df['Miners Supportable'].sum():,.0f} mining containers could run during surplus windows")